# Env Setup

In [5]:
import requests
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, round, desc
from pyspark.sql.types import StructType, StructField, StringType

# 1. Load Secrets
load_dotenv()
CLIENT_ID = os.getenv("OPENSKY_CLIENT_ID")
CLIENT_SECRET = os.getenv("OPENSKY_CLIENT_SECRET")

# 2. Initialize Local Spark Session (Optimized for Delta Lake)
spark = SparkSession.builder \
    .appName("Aviation_Density_Local") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.driver.host", "localhost") \
    .getOrCreate()

print("✅ Spark Session Initialized with Delta Lake Support")

✅ Spark Session Initialized with Delta Lake Support


## BRONZE LAYER (Data Ingestion)

In [6]:
# 1. Get OAuth2 Access Token
def get_opensky_token(client_id, client_secret):
    token_url = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"
    data = {"grant_type": "client_credentials", "client_id": client_id, "client_secret": client_secret}
    
    response = requests.post(token_url, data=data)
    if response.status_code == 200:
        return response.json().get("access_token")
    else:
        raise Exception(f"Failed to get token: {response.status_code} - {response.text}")

token = get_opensky_token(CLIENT_ID, CLIENT_SECRET)

# 2. Fetch API Data
url = "https://opensky-network.org/api/states/all"
headers = {"Authorization": f"Bearer {token}"}
response = requests.get(url, headers=headers)

if response.status_code == 200:
    flight_vectors = response.json().get("states", [])[:2000]
    print(f"✅ Success! Retrieved {len(flight_vectors)} flights.")
else:
    raise Exception(f"❌ API Error {response.status_code}: {response.text}")

# 3. Stringify Data & Define Schema
safe_data = [[str(i) if i is not None else None for i in f] for f in flight_vectors]

bronze_schema = StructType([
    StructField("icao24", StringType(), True), StructField("callsign", StringType(), True),
    StructField("origin_country", StringType(), True), StructField("time_position", StringType(), True),
    StructField("last_contact", StringType(), True), StructField("longitude", StringType(), True),
    StructField("latitude", StringType(), True), StructField("baro_altitude", StringType(), True),
    StructField("on_ground", StringType(), True), StructField("velocity", StringType(), True),
    StructField("true_track", StringType(), True), StructField("vertical_rate", StringType(), True),
    StructField("sensors", StringType(), True), StructField("geo_altitude", StringType(), True),
    StructField("squawk", StringType(), True), StructField("spi", StringType(), True),
    StructField("position_source", StringType(), True)
])

# 4. Write BRONZE (Append Mode)
df_bronze = spark.createDataFrame(safe_data, schema=bronze_schema)
df_bronze.write.format("delta").mode("append").save("./data/bronze_flights")
print("✅ Bronze layer saved safely!")

✅ Success! Retrieved 2000 flights.
✅ Bronze layer saved safely!


## SILVER LAYER (Transformation & Cleansing)

In [ ]:
# 1. Read from Bronze
df_bronze_loaded = spark.read.format("delta").load("./data/bronze_flights")

# 2. Transform (Cast and Filter)
df_silver = df_bronze_loaded.select(
    col("icao24"),
    col("callsign"),
    col("origin_country"),              
    col("longitude").cast("float"),      
    col("latitude").cast("float"),
    col("baro_altitude").cast("float"),
    col("velocity").cast("float"),
    col("true_track").cast("float"),
    col("on_ground").cast("boolean")     
).filter(
    (col("on_ground") == False) &          
    (col("latitude").isNotNull()) &        
    (col("longitude").isNotNull()) &
    (col("baro_altitude").isNotNull()) &   
    (col("velocity") > 0)                  
)

# 3. Write SILVER (Overwrite Mode with Schema Evolution)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("./data/silver_flights")

print("✅ Silver layer cleaned, casted, and saved!")
df_silver.show(5)

26/04/22 17:08:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


✅ Silver layer cleaned, casted, and saved!
+------+--------+--------------+---------+--------+-------------+--------+----------+---------+
|icao24|callsign|origin_country|longitude|latitude|baro_altitude|velocity|true_track|on_ground|
+------+--------+--------------+---------+--------+-------------+--------+----------+---------+
|8014cf|IGO9270 |         India|  88.6757| 22.5999|      3246.12|  151.29|     91.36|    false|
|344459|AEA47WJ |         Spain| -15.2286| 29.8801|     11117.58|  229.16|     214.3|    false|
|a474e8|N3867G  | United States| -81.1835|  35.785|      1463.04|   97.84|     65.46|    false|
|471f68|WMT5GS  |       Hungary|  24.5135| 38.9564|     10241.28|   178.7|    177.36|    false|
|4a2022|ROT416M |       Romania|  12.6669| 41.8243|      10972.8|  232.81|     72.25|    false|
+------+--------+--------------+---------+--------+-------------+--------+----------+---------+
only showing top 5 rows



## GOLD LAYER (Business Aggregates)

In [8]:
# 1. Read from Silver
df_silver_loaded = spark.read.format("delta").load("./data/silver_flights")

# 2. Aggregate
df_gold = df_silver_loaded.groupBy("origin_country") \
    .agg(
        count("icao24").alias("flight_count"),
        round(avg("velocity"), 2).alias("avg_velocity_ms"),
        round(avg("baro_altitude"), 2).alias("avg_altitude_m")
    ) \
    .filter(col("flight_count") > 5) \
    .orderBy(desc("flight_count"))

# 3. Write GOLD (Overwrite Mode with Schema Evolution)
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("./data/gold_aviation_metrics")

print("✅ Gold layer (Analytical Aggregates) saved!")
df_gold.show(10)

✅ Gold layer (Analytical Aggregates) saved!
+--------------------+------------+---------------+--------------+
|      origin_country|flight_count|avg_velocity_ms|avg_altitude_m|
+--------------------+------------+---------------+--------------+
|       United States|        7609|         149.93|       5594.32|
|              Canada|         523|         166.57|       6860.29|
|              Turkey|         431|         219.77|       9664.69|
|               India|         346|         208.29|       8645.75|
|United Arab Emirates|         331|         236.61|        9292.3|
|               Spain|         281|         187.51|       7234.12|
|               China|         235|         221.99|       8835.34|
|      United Kingdom|         226|         160.97|       6703.68|
|              Brazil|         224|         172.49|       6668.21|
|             Austria|         205|          190.9|        7332.3|
+--------------------+------------+---------------+--------------+
only showing top 1

## Pipeline Teardown

In [9]:
# Clean Shutdown
spark.stop()
print("🛑 Spark Session stopped gracefully.")

🛑 Spark Session stopped gracefully.
